In [ ]:
import json
import time
from functools import lru_cache
from pathlib import Path
from types import SimpleNamespace

import cv2
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from keras import layers
from keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
print("TensorFlow :", tf.__version__)
print("GPU détecté :", tf.config.list_physical_devices("GPU"))

In [ ]:
# --- Trouver automatiquement le dossier qui contient glioma/, meningioma/, ... ---
def find_raw_dir(base=Path("/kaggle/input")):
    found = [p.parent for p in base.rglob("glioma") if p.is_dir()]
    if not found:
        raise FileNotFoundError("Dossier 'glioma' introuvable : vérifie que le dataset est bien ajouté au notebook.")
    if len(found) > 1:
        print(" Plusieurs dossiers trouvés :", found, "-> le premier est utilisé. Modifie RAW_DIR à la main si besoin.")
    return found[0]


RAW_DIR = find_raw_dir()
print("Dataset trouvé dans :", RAW_DIR)

WORK = Path("/kaggle/working")          # seul dossier où Kaggle autorise l'écriture

config = SimpleNamespace(
    RAW_DIR=RAW_DIR,
    CACHE_FILE=WORK / "processed" / "dataset.npz",
    MODELS_DIR=WORK / "models",
    REPORTS_DIR=WORK / "reports",
    MODEL_PATH=WORK / "models" / "best_model.keras",
    BACKUP_DIR=WORK / "models" / "backup",

    CLASSES=["glioma", "meningioma", "notumor", "pituitary"],
    VALID_EXT={".jpeg", ".jpg", ".bmp", ".png"},
    IMG_SIZE=224,

    SEED=42,
    BATCH_SIZE=32,
    EPOCHS=40,
    LEARNING_RATE=1e-3,
)

In [ ]:
# 1) Chargement brut des images avec OpenCV
def load_images(raw_dir=config.RAW_DIR, img_size=config.IMG_SIZE):
    images, labels = [], []
    skipped = 0

    for class_dir in sorted(raw_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label = class_dir.name

        for file in class_dir.iterdir():
            if file.suffix.lower() not in config.VALID_EXT:
                skipped += 1
                continue

            try:
                img = cv2.imread(str(file), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    raise ValueError("image illisible")
                img = cv2.resize(img, (img_size, img_size))
            except Exception as e:
                print(f"[ignorée] {file.name} : {e}")
                skipped += 1
                continue

            images.append(img)
            labels.append(label)

    print(f"{len(images)} images chargées, {skipped} ignorées.")


    images, labels = shuffle(images, labels, random_state=config.SEED)


    X = np.array(images, dtype="uint8")[..., np.newaxis]
    y = np.array(labels)
    return X, y


def load_dataset(use_cache=True):
    if use_cache and config.CACHE_FILE.exists():
        data = np.load(config.CACHE_FILE)
        return data["X"], data["y"]

    X, y = load_images()
    config.CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(config.CACHE_FILE, X=X, y=y)
    return X, y


# 2) Étiquettes, normalisation, split
def encode_labels(y):
    encoder = LabelEncoder().fit(config.CLASSES)
    return encoder.transform(y)


def normalize(image):
    return tf.cast(image, tf.float32) / 255.0


def get_splits():
    X, y = load_dataset()
    y_int = encode_labels(y)

    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y_int, test_size=0.30, stratify=y_int, random_state=config.SEED
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=config.SEED
    )


    n = len(config.CLASSES)
    return (X_train, to_categorical(y_train, n),
            X_val, to_categorical(y_val, n),
            X_test, to_categorical(y_test, n))


def get_class_weights(y_train_onehot):
    y_int = np.argmax(y_train_onehot, axis=1)
    classes = np.unique(y_int)
    weights = compute_class_weight("balanced", classes=classes, y=y_int)
    return {int(c): float(w) for c, w in zip(classes, weights)}


# 3) Pipeline tf.data : cache + préchargement
def make_dataset(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.cache()
    if training:
        ds = ds.shuffle(len(X), seed=config.SEED)
    ds = ds.map(lambda img, lbl: (normalize(img), lbl),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(config.BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
X, y = load_dataset()
print("X:", X.shape, X.dtype, "| y:", y.shape)

classes, counts = np.unique(y, return_counts=True)
sns.barplot(x=classes, y=counts)
plt.title("Nombre d'images par classe")
plt.show()
print(dict(zip(classes, counts)))

In [ ]:
fig, axes = plt.subplots(len(config.CLASSES), 4, figsize=(10, 10))
for row, cls in enumerate(config.CLASSES):
    idx = np.where(y == cls)[0][:4]
    for col, i in enumerate(idx):
        axes[row, col].imshow(X[i].squeeze(), cmap="gray")
        axes[row, col].axis("off")
        axes[row, col].set_title(cls, fontsize=9)
plt.show()

In [ ]:
img = X[0].squeeze()
norm = img.astype("float32") / 255.0

fig, axes = plt.subplots(2, 2, figsize=(9, 7))
axes[0, 0].imshow(img, cmap="gray");  axes[0, 0].set_title(f"Avant : {img.min()} à {img.max()}")
axes[0, 1].imshow(norm, cmap="gray"); axes[0, 1].set_title(f"Après : {norm.min():.2f} à {norm.max():.2f}")
axes[1, 0].hist(img.ravel(), bins=50);  axes[1, 0].set_title("Histogramme avant")
axes[1, 1].hist(norm.ravel(), bins=50); axes[1, 1].set_title("Histogramme après")
plt.show()

In [ ]:
aug = keras.Sequential([layers.RandomRotation(0.05), layers.RandomZoom(0.1),
                        layers.RandomTranslation(0.05, 0.05)])
sample = (X[0:1] / 255.0).astype("float32")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(sample[0].squeeze(), cmap="gray"); axes[0].set_title("Original")
for ax in axes[1:]:
    ax.imshow(aug(sample, training=True)[0].numpy().squeeze(), cmap="gray")
    ax.set_title("Augmentée")
plt.show()

In [ ]:
def build_model():
    augmentation = keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.05, 0.05),
    ], name="augmentation")

    inputs = keras.Input(shape=(config.IMG_SIZE, config.IMG_SIZE, 1))
    x = augmentation(inputs)


    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(len(config.CLASSES), activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="neurovision_cnn")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
def plot_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(history.history[metric], label="train")
        ax.plot(history.history[f"val_{metric}"], label="validation")
        ax.set_title(metric)
        ax.set_xlabel("époque")
        ax.legend()
    fig.savefig(config.REPORTS_DIR / "learning_curves.png", dpi=150, bbox_inches="tight")


def train_main():
    config.MODELS_DIR.mkdir(exist_ok=True)
    config.REPORTS_DIR.mkdir(exist_ok=True)
    keras.utils.set_random_seed(config.SEED)

    # Données
    X_train, y_train, X_val, y_val, X_test, y_test = get_splits()
    print("train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)
    class_weights = get_class_weights(y_train)
    print("class_weight:", class_weights)

    train_ds = make_dataset(X_train, y_train, training=True)
    val_ds = make_dataset(X_val, y_val)


    model = build_model()
    model.summary()
    try:
        keras.utils.plot_model(model, to_file=str(config.REPORTS_DIR / "model.png"),
                               show_shapes=True)
    except Exception as e:
        print("plot_model ignoré :", e)


    callbacks = [
        keras.callbacks.ModelCheckpoint(
            config.MODEL_PATH, monitor="val_accuracy", save_best_only=True),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True),
        keras.callbacks.BackupAndRestore(
            backup_dir=str(config.BACKUP_DIR)),
    ]

    start = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=config.EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks,
    )
    duration = time.time() - start
    print(f"Durée d'entraînement : {duration / 60:.1f} min")

    plot_curves(history)
    with open(config.REPORTS_DIR / "history.json", "w") as f:
        json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f)

In [ ]:
train_main()

In [ ]:
def evaluate_main():
    *_, X_test, y_test = get_splits()
    model = keras.models.load_model(config.MODEL_PATH)

    test_ds = make_dataset(X_test, y_test, training=False)
    loss, acc = model.evaluate(test_ds)
    print(f"Test -> loss: {loss:.4f} | accuracy: {acc:.4f}")

    y_true = np.argmax(y_test, axis=1)
    y_pred = np.argmax(model.predict(test_ds), axis=1)


    print(classification_report(y_true, y_pred, target_names=config.CLASSES))


    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=config.CLASSES, yticklabels=config.CLASSES)
    plt.xlabel("Prédit")
    plt.ylabel("Réel")
    plt.title("Matrice de confusion")
    plt.savefig(config.REPORTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")


    for name, mask in [("correct", y_true == y_pred), ("incorrect", y_true != y_pred)]:
        idx = np.where(mask)[0][:8]
        if len(idx) == 0:
            continue
        fig, axes = plt.subplots(2, 4, figsize=(12, 6))
        for ax, i in zip(axes.ravel(), idx):
            ax.imshow(X_test[i].squeeze(), cmap="gray")
            ax.set_title(f"réel: {config.CLASSES[y_true[i]]}\nprédit: {config.CLASSES[y_pred[i]]}",
                         fontsize=9)
        for ax in axes.ravel():
            ax.axis("off")
        fig.savefig(config.REPORTS_DIR / f"examples_{name}.png", dpi=150, bbox_inches="tight")

In [ ]:
evaluate_main()

In [ ]:
@lru_cache(maxsize=1)
def get_model():
    if not config.MODEL_PATH.exists():
        raise FileNotFoundError(f"Modèle introuvable : {config.MODEL_PATH}. Lance d'abord l'entraînement.")
    return keras.models.load_model(config.MODEL_PATH)


def predict_array(img_gray):
    img = cv2.resize(img_gray, (config.IMG_SIZE, config.IMG_SIZE))
    img = img.astype("float32") / 255.0
    img = img[np.newaxis, ..., np.newaxis]

    probs = get_model().predict(img, verbose=0)[0]
    best = int(np.argmax(probs))
    return config.CLASSES[best], float(probs[best]), dict(zip(config.CLASSES, map(float, probs)))


def predict_image(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    if path.suffix.lower() not in config.VALID_EXT:
        raise ValueError(f"Extension non supportée : {path.suffix}")

    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError("Image illisible ou corrompue.")
    return predict_array(img)

In [ ]:
test_path = next((config.RAW_DIR / "glioma").iterdir())
try:
    label, conf, probs = predict_image(test_path)
    print(f"{label} ({conf:.1%})")
    print(probs)
except Exception as e:
    print("Erreur :", e)